In [ ]:
# ============================================================
# Datalog Examples with:
#   - EDB (facts) across categories
#   - IDB (rules) across categories
#   - Recursion + fixpoint intuition
#   - Negation + stratified negation (safe patterns)
#   - Hard constraints (forbid/override actions)
#   - Explanation hooks: traceable “why” facts
#
# Domain: after-hours pediatric triage (CareTrace-style)
#
# NOTE: This is an educational KB + rule set. Not medical advice !
# ============================================================

In [1]:
# If needed:
!pip -q install pydatalog

from pyDatalog import pyDatalog

### ALL Terms must be defined !

In [36]:
# ============================================================
# 0) TERMS: variables + predicate symbols
# ============================================================

pyDatalog.clear()

pyDatalog.create_terms(
    # Variables
    'P, X, Y, Z, S, Dx, Med, Cond, Test, Val, Level, Loc, T, Reason, Action, Obs, Age',

    # Patient profile / context (EDB)
    'age_years, weight_lb, time_of_day, day_of_illness, caregiver_goal',
    'temp_f, temp_method, heart_rate, resp_rate, spo2',
    'appears, pain_location, pain_severity, pain_with_jump',
    'vomit_count_24h, diarrhea_count_24h, tolerates_fluids',
    'urine_count_since_morning, hours_since_last_urine, dry_mucosa, tears_present',
    'rash_type, neck_stiffness, mental_status',
    'comorbidity, allergy, medication_taken, recent_exposure',
    'local_signal', 'cough', 'no_cough',

    # Clinical concept layer (EDB)
    'symptom, sign, red_flag_symptom, red_flag_sign',
    'threshold, rule_priority',

    # Guideline/policy knowledge (EDB)
    'guideline',
    'antibiotic_indicated_for, test_recommended_for, condition_clues',

    # Medication knowledge (EDB)
    'med_class, contraindicated_if, avoid_if, dose_mg_per_kg, max_mg_per_kg_day, interval_hours',
    'dose_mg_per_lb, max_doses_per_day',

    # Derived states (IDB)
    'dehydration_level, dehydration_feature_count, has_red_flag, likely_appendicitis_pattern',
    'likely_viral_gi_pattern, needs_escalation, escalation_level',
    'recommended_action, forbidden_action, recommended_med, avoid_med',
    'needs_test, antibiotic_plan', 'eligible_home_care',

    # Evidence / explanation (IDB)
    'because, supports, supports_chain, explain_line',

    # Recursion (IDB)
    'edge, reach',

    #States and Cases (IDB)
    'risk_state, case'
)


### The EDB: Our "Ground Database"

#### EDB: Patient Cases 

In [ ]:
# ============================================================
# 1) EDB: PATIENT CASES (richer than a toy)
#    We’ll define 3 cases:
#      - childA: “11pm fever + vomiting + low urine” (your main)
#      - childB: RLQ pain pattern (appendicitis concern)
#      - childC: respiratory red flag (hypoxia)
# ============================================================

# --- Case A: after-hours caregiver scenario ---
+age_years('childA', 6)
+weight_lb('childA', 44)
+time_of_day('childA', '23:00')
+day_of_illness('childA', 2)
+caregiver_goal('childA', 'safe_plan_tonight_avoid_er_if_reasonable')

+temp_f('childA', 102.6)
+temp_method('childA', 'forehead')
+vomit_count_24h('childA', 1)
+diarrhea_count_24h('childA', 0)
+tolerates_fluids('childA', 'limited_sips')
+urine_count_since_morning('childA', 1)
+hours_since_last_urine('childA', 7)
+dry_mucosa('childA', True)
+tears_present('childA', False)
+mental_status('childA', 'tired_but_arousable')
+appears('childA', 'wiped_out')
+pain_location('childA', 'diffuse_abdomen')
+pain_severity('childA', 'mild_to_moderate')
+pain_with_jump('childA', False)

+medication_taken('childA', 'none_today')
+allergy('childA', 'none_known')
+comorbidity('childA', 'none')

+local_signal('childA', 'norovirus_school_wave')  # synthetic “context” channel

# --- Case B: appendicitis-like pattern ---
+age_years('childB', 7)
+weight_lb('childB', 52)
+time_of_day('childB', '22:30')
+day_of_illness('childB', 1)

+temp_f('childB', 101.8)
+vomit_count_24h('childB', 3)
+tolerates_fluids('childB', 'poor')
+urine_count_since_morning('childB', 1)
+hours_since_last_urine('childB', 9)
+mental_status('childB', 'ill_appearing')

+pain_location('childB', 'RLQ')
+pain_severity('childB', 'severe')
+pain_with_jump('childB', True)
+tears_present('childB', False)
+dry_mucosa('childB', True)

# --- Case C: respiratory red flag ---
+age_years('childC', 5)
+weight_lb('childC', 40)
+time_of_day('childC', '00:15')
+temp_f('childC', 103.4)
+resp_rate('childC', 40)
+spo2('childC', 88)
+appears('childC', 'distressed')
+mental_status('childC', 'sleepy_hard_to_wake')

#### EDB: Medical Knowledge

In [22]:
# ============================================================
# 2) EDB: CLINICAL CONCEPTS + RED FLAGS + THRESHOLDS
# ============================================================

# Red-flag symptoms/signs (simplified set)
+red_flag_sign('spo2_low')
+red_flag_sign('resp_distress')
+red_flag_symptom('cyanosis_blue_lips')
+red_flag_symptom('unresponsive')
+red_flag_symptom('nonblanching_purple_rash')
+red_flag_symptom('severe_localized_RLQ_pain')
+red_flag_symptom('bilious_vomiting')
+red_flag_symptom('neck_stiffness')

# Thresholds: encode key cutoffs as facts (lets you show “authority”)
+threshold('spo2_emergency_below', 90)
+threshold('no_urine_hours_redflag', 10)
+threshold('no_urine_hours_concern', 8)
+threshold('vomit_count_persistent', 3)
+threshold('temp_high_f', 104.0)

# Rule priorities (when conflicts arise)
+rule_priority('emergency', 100)
+rule_priority('urgent', 70)
+rule_priority('home', 30)

# ============================================================
# 3) EDB: MEDICATION + GUIDELINE KNOWLEDGE (compact but real-ish)
# ============================================================

# Medication classes
+med_class('acetaminophen', 'antipyretic')
+med_class('ibuprofen', 'nsaid')

# Dosing facts (teaching-oriented)
+dose_mg_per_kg('acetaminophen', 15)
+interval_hours('acetaminophen', 4)          # typical 4–6; we use 4 as lower bound
+max_mg_per_kg_day('acetaminophen', 75)
+max_doses_per_day('acetaminophen', 5)

+dose_mg_per_kg('ibuprofen', 10)
+interval_hours('ibuprofen', 6)
+max_mg_per_kg_day('ibuprofen', 40)
+max_doses_per_day('ibuprofen', 4)

# Contraindications / avoid conditions (illustrative)
+avoid_if('ibuprofen', 'dehydration_risk')
+avoid_if('ibuprofen', 'kidney_risk')
+avoid_if('ibuprofen', 'persistent_vomiting')

# Antibiotics guidance (teaching scope)
+antibiotic_indicated_for('strep_pharyngitis_confirmed')
+test_recommended_for('strep_pharyngitis_suspected', 'rapid_strep_or_culture')
+condition_clues('strep_pharyngitis_suspected', 'fever')
+condition_clues('strep_pharyngitis_suspected', 'abdominal_pain')
+condition_clues('strep_pharyngitis_suspected', 'no_cough')


In [23]:
# Just simple queries
# Please note that variables like X and T have to be explicitly defined as pytDatalog terms

print(age_years('childA', X)) 
print(temp_f('childB',T))

X
-
6
T    
-----
101.8


### IDB: The Inferred Database

In [29]:
# ============================================================
# 4) IDB: DERIVE NORMALIZED “OBSERVATIONS” (map patient facts → signs/symptoms)
#    This lets our rules operate over a consistent vocabulary.
# ============================================================

# High fever signal
sign(P, 'temp_high') <= (temp_f(P, T) & (T >= 104.0))

# Spo2 low (red flag sign)
sign(P, 'spo2_low') <= (spo2(P, Val) & threshold('spo2_emergency_below', Z) & (Val < Z))

# Respiratory distress proxy
sign(P, 'resp_distress') <= (resp_rate(P, Val) & (Val >= 35))

# Severe RLQ pain symptom
symptom(P, 'severe_localized_RLQ_pain') <= (pain_location(P, 'RLQ') & pain_severity(P, 'severe'))

# Persistent vomiting symptom
symptom(P, 'persistent_vomiting') <= (vomit_count_24h(P, Val) & threshold('vomit_count_persistent', Z) & (Val >= Z))

# Dehydration features
symptom(P, 'low_urine') <= (hours_since_last_urine(P, Val) & threshold('no_urine_hours_concern', Z) & (Val >= Z))
symptom(P, 'very_low_urine') <= (hours_since_last_urine(P, Val) & threshold('no_urine_hours_redflag', Z) & (Val >= Z))
symptom(P, 'dry_mucosa') <= dry_mucosa(P, True)
symptom(P, 'no_tears') <= (tears_present(P, False))
symptom(P, 'poor_fluids') <= (tolerates_fluids(P, 'poor'))

# ============================================================
# 5) IDB: DEHYDRATION ASSESSMENT (multi-factor, explainable)
# ============================================================

# Count dehydration features 
dehydration_feature_count(P, 1) <= symptom(P, 'dry_mucosa') & ~symptom(P, 'no_tears') & ~symptom(P, 'low_urine') & ~symptom(P, 'poor_fluids')
dehydration_feature_count(P, 2) <= symptom(P, 'dry_mucosa') & symptom(P, 'no_tears') & ~symptom(P, 'low_urine') & ~symptom(P, 'poor_fluids')
dehydration_feature_count(P, 2) <= symptom(P, 'low_urine') & symptom(P, 'dry_mucosa') & ~symptom(P, 'no_tears')
dehydration_feature_count(P, 3) <= symptom(P, 'low_urine') & symptom(P, 'dry_mucosa') & symptom(P, 'no_tears')
dehydration_feature_count(P, 4) <= symptom(P, 'very_low_urine') & symptom(P, 'dry_mucosa') & symptom(P, 'no_tears') & symptom(P, 'poor_fluids')

# Dehydration level derived from feature count 
dehydration_level(P, 'mild') <= dehydration_feature_count(P, X) & (X == 1)
dehydration_level(P, 'moderate') <= dehydration_feature_count(P, X) & (X == 2)
dehydration_level(P, 'moderate') <= dehydration_feature_count(P, X) & (X == 3)
dehydration_level(P, 'severe') <= dehydration_feature_count(P, X) & (X >= 4)

# Evidence statements for explainability
because(P, 'dehydration_risk', 'dry_mucosa') <= symptom(P, 'dry_mucosa')
because(P, 'dehydration_risk', 'no_tears') <= symptom(P, 'no_tears')
because(P, 'dehydration_risk', 'low_urine') <= symptom(P, 'low_urine')
because(P, 'dehydration_risk', 'very_low_urine') <= symptom(P, 'very_low_urine')
because(P, 'dehydration_risk', 'poor_fluids') <= symptom(P, 'poor_fluids')

# ============================================================
# 6) IDB: RED FLAGS + ESCALATION LOGIC (with priorities + safety-netting)
# ============================================================

has_red_flag(P) <= sign(P, 'spo2_low')
has_red_flag(P) <= sign(P, 'resp_distress')
has_red_flag(P) <= symptom(P, 'severe_localized_RLQ_pain')
has_red_flag(P) <= symptom(P, 'very_low_urine')
has_red_flag(P) <= (mental_status(P, 'sleepy_hard_to_wake'))

# Escalation levels (Emergency > Urgent > Home)
escalation_level(P, 'emergency') <= sign(P, 'spo2_low')
escalation_level(P, 'emergency') <= (mental_status(P, 'sleepy_hard_to_wake'))
escalation_level(P, 'urgent') <= symptom(P, 'very_low_urine')
escalation_level(P, 'urgent') <= symptom(P, 'severe_localized_RLQ_pain')
escalation_level(P, 'urgent') <= dehydration_level(P, 'severe')
escalation_level(P, 'home') <= (~has_red_flag(P) & dehydration_level(P, 'mild'))
escalation_level(P, 'home') <= (~has_red_flag(P) & dehydration_level(P, 'moderate'))

needs_escalation(P) <= has_red_flag(P)
needs_escalation(P) <= escalation_level(P, 'urgent')
needs_escalation(P) <= escalation_level(P, 'emergency')

# ============================================================
# 7) IDB: DIAGNOSTIC PATTERN RULES (appendicitis vs viral GI)
# ============================================================

# Appendicitis-like: RLQ + jump pain + vomiting + ill appearing
likely_appendicitis_pattern(P) <= (
    pain_location(P, 'RLQ') &
    pain_with_jump(P, True) &
    symptom(P, 'persistent_vomiting')
)

likely_appendicitis_pattern(P) <= (
    pain_location(P, 'RLQ') &
    pain_with_jump(P, True) &
    (mental_status(P, 'ill_appearing'))
)

because(P, 'appendicitis_concern', 'rlq_pain') <= pain_location(P, 'RLQ')
because(P, 'appendicitis_concern', 'jump_pain') <= pain_with_jump(P, True)
because(P, 'appendicitis_concern', 'persistent_vomiting') <= symptom(P, 'persistent_vomiting')

# Viral GI-like: vomiting +/- diarrhea + diffuse pain + local signal
likely_viral_gi_pattern(P) <= (
    (vomit_count_24h(P, X) & (X >= 1)) &
    pain_location(P, 'diffuse_abdomen') &
    local_signal(P, 'norovirus_school_wave')
)

because(P, 'viral_gi_support', 'local_signal_norovirus') <= local_signal(P, 'norovirus_school_wave')
because(P, 'viral_gi_support', 'diffuse_pain') <= pain_location(P, 'diffuse_abdomen')
because(P, 'viral_gi_support', 'vomiting') <= (vomit_count_24h(P, X) & (X >= 1))

# ============================================================
# 8) IDB: ACTION PLANNING (plan tonight + crisp thresholds)
# ============================================================

# Emergency action
recommended_action(P, 'go_to_er_now') <= escalation_level(P, 'emergency')

# Urgent action
recommended_action(P, 'urgent_same_night_evaluation') <= (
    escalation_level(P, 'urgent') & ~escalation_level(P, 'emergency')
)

# Home plan allowed only if not emergency/urgent and mild/moderate dehydration
recommended_action(P, 'home_plan_with_monitoring') <= (
    escalation_level(P, 'home') & ~escalation_level(P, 'urgent') & ~escalation_level(P, 'emergency')
)

# Hard constraint: if emergency or urgent, forbid "home-only"
forbidden_action(P, 'home_plan_with_monitoring') <= escalation_level(P, 'urgent')
forbidden_action(P, 'home_plan_with_monitoring') <= escalation_level(P, 'emergency')

# Safety-netting: generate explicit thresholds as actions
recommended_action(P, 'monitor_urine_output_threshold') <= dehydration_level(P, Level)
recommended_action(P, 'small_frequent_sips_ors') <= dehydration_level(P, Level)
recommended_action(P, 'return_precautions_red_flags') <= age_years(P, Age)

# ============================================================
# 9) IDB: MED SELECTION (grounded in risk states + constraints)
# ============================================================

# Risk state facts derived from clinical assessment
+edge('dehydration_risk', 'kidney_risk')  # a “knowledge edge” we can traverse

# Derive dehydration_risk if dehydration is moderate or worse OR low urine present
risk_state(P, 'dehydration_risk') <= dehydration_level(P, 'moderate')
risk_state(P, 'dehydration_risk') <= dehydration_level(P, 'severe')
risk_state(P, 'dehydration_risk') <= symptom(P, 'low_urine')
risk_state(P, 'persistent_vomiting') <= symptom(P, 'persistent_vomiting')

# Recursive propagation of risks via edges (shows “graph semantics” in logic)
reach(X, Y) <= edge(X, Y)
reach(X, Y) <= (edge(X, Z) & reach(Z, Y))

risk_state(P, 'kidney_risk') <= (risk_state(P, 'dehydration_risk') & reach('dehydration_risk', 'kidney_risk'))

# Avoid meds by contraindication rules
avoid_med(P, Med) <= (avoid_if(Med, Cond) & risk_state(P, Cond))

# Prefer acetaminophen for dehydration risk; ibuprofen only if not in avoid list
recommended_med(P, 'acetaminophen') <= risk_state(P, 'dehydration_risk')
recommended_med(P, 'ibuprofen') <= (~avoid_med(P, 'ibuprofen') & ~risk_state(P, 'dehydration_risk'))

# If both could apply, keep acetaminophen as default (simple precedence)
recommended_med(P, 'acetaminophen') <= (~avoid_med(P, 'acetaminophen') & ~recommended_med(P, 'ibuprofen'))



recommended_med(P,'acetaminophen') <= ~avoid_med(P

In [25]:
# ============================================================
# 10) IDB: TESTING + ANTIBIOTIC PLAN (guideline-ish and scoped)
# ============================================================

# NOTE:
# - pyDatalog does NOT support (Query | Query) for OR.
# - To express OR, write multiple rules with the same head.
# - For negation (~...), bind variables via a base predicate like case(P)
#   so the rule is "safe" (range-restricted).

# Define what counts as a known case/patient in the KB (anchor predicate).
# Any always-present EDB predicate works; age_years is a good anchor.
_ = case(P) <= age_years(P, Age)

# -------------------------------
# Strep suspicion heuristic (illustrative):
# fever + abdominal pain + no cough
# -------------------------------

# Derive symptom 'abdominal_pain' from pain severity (OR via two rules)
_ = symptom(P, 'abdominal_pain') <= pain_severity(P, 'mild_to_moderate')
_ = symptom(P, 'abdominal_pain') <= pain_severity(P, 'severe')

# "No cough" as absence of cough (stratified negation),
# anchored by case(P) so P is bound.
_ = symptom(P, 'no_cough') <= case(P) & ~symptom(P, 'cough')

# Needs test if abdominal pain + fever threshold + no cough
_ = needs_test(P, 'rapid_strep_or_culture') <= (
    symptom(P, 'abdominal_pain') &
    temp_f(P, T) & (T >= 100.4) &
    symptom(P, 'no_cough')
)

# Antibiotic plan depends on confirmed strep (we do not confirm in this notebook),
# so we recommend not prescribing without confirmation.
_ = antibiotic_plan(P, 'no_antibiotics_without_confirmation') <= needs_test(P, 'rapid_strep_or_culture')


# ============================================================
# 11) EXPLANATION / EVIDENCE ASSEMBLY (stable across turns)
# ============================================================

# supports(): map derived conclusions to supporting facts
_ = supports(P, 'needs_escalation', 'spo2_low') <= sign(P, 'spo2_low')
_ = supports(P, 'needs_escalation', 'very_low_urine') <= symptom(P, 'very_low_urine')
_ = supports(P, 'needs_escalation', 'severe_RLQ_pain') <= symptom(P, 'severe_localized_RLQ_pain')
_ = supports(P, 'needs_escalation', 'hard_to_wake') <= mental_status(P, 'sleepy_hard_to_wake')

_ = supports(P, 'dehydration_risk', 'dry_mucosa') <= symptom(P, 'dry_mucosa')
_ = supports(P, 'dehydration_risk', 'no_tears') <= symptom(P, 'no_tears')
_ = supports(P, 'dehydration_risk', 'low_urine') <= symptom(P, 'low_urine')

_ = supports(P, 'appendicitis_concern', 'rlq_pain') <= pain_location(P, 'RLQ')
_ = supports(P, 'appendicitis_concern', 'jump_pain') <= pain_with_jump(P, True)
_ = supports(P, 'appendicitis_concern', 'persistent_vomiting') <= symptom(P, 'persistent_vomiting')

# Recursive support chaining for “multi-hop rationale” (graph of justification)
_ = supports_chain(P, X, Y) <= supports(P, X, Y)
_ = supports_chain(P, X, Y) <= (supports(P, X, Z) & supports_chain(P, Z, Y))

# Canonical “explain lines”
_ = explain_line(P, 'escalation', Reason) <= supports(P, 'needs_escalation', Reason)
_ = explain_line(P, 'dehydration', Reason) <= supports(P, 'dehydration_risk', Reason)
_ = explain_line(P, 'appendicitis', Reason) <= supports(P, 'appendicitis_concern', Reason)


# -------------------------------
# Demo queries (run after your facts are asserted)
# -------------------------------

print("Derived symptoms for childA:")
print(symptom('childA', X))

print("\nNeeds test?")
print(needs_test('childA', X))

print("\nAntibiotic plan:")
print(antibiotic_plan('childA', X))

print("\nExplain lines (any that fire):")
print(explain_line('childA', X, Y))


Derived symptoms for childA:
X             
--------------
no_cough      
abdominal_pain
no_tears      
dry_mucosa    

Needs test?
X                     
----------------------
rapid_strep_or_culture

Antibiotic plan:
X                                  
-----------------------------------
no_antibiotics_without_confirmation

Explain lines (any that fire):
X           | Y         
------------|-----------
dehydration | no_tears  
dehydration | dry_mucosa


In [30]:
# -------------------------------
# Better Demo Queries (shows each capability clearly)
# -------------------------------

print("\n=== 1) EDB sanity check (raw facts) ===")
print("age_years:", age_years('childA', X))
print("temp_f:", temp_f('childA', T))
print("pain_severity:", pain_severity('childA', X))

print("\n=== 2) IDB derivation (new facts from rules) ===")
print("derived symptom(childA, ...):", symptom('childA', X))
print("derived symptom(childA,'abdominal_pain'):", symptom('childA', 'abdominal_pain'))

print("\n=== 3) OR-across-rules demonstration ===")
print("abdominal_pain can come from mild_to_moderate OR severe")
print("pain_severity(childA,'mild_to_moderate') -> abdominal_pain:",
      symptom('childA', 'abdominal_pain'))

print("\n=== 4) Stratified negation demonstration (no_cough) ===")
print("no cough present in facts, so symptom(childA,'no_cough') derives:")
print("no_cough:", symptom('childA', 'no_cough'))

# Optional: uncomment these 2 lines during lecture to show the flip
# +symptom('childA', 'cough')
# print("after adding cough, no_cough:", symptom('childA', 'no_cough'))

print("\n=== 5) Conjunction + numeric comparison (fever threshold) ===")
print("needs_test(childA, X):", needs_test('childA', X))

print("\n=== 6) A derived conclusion depending on another derived conclusion ===")
print("antibiotic_plan(childA, X):", antibiotic_plan('childA', X))

print("\n=== 7) Explanation mapping (supports -> explain_line) ===")
print("explain_line(childA, Category, Reason):", explain_line('childA', X, Y))

print("\n=== 8) Recursion / transitive closure over justification (supports_chain) ===")
print("supports_chain(childA, X, Y):", supports_chain('childA', X, Y))

print("\n=== 9) Focused: show multi-hop chain if it exists ===")
print("supports(childA,'needs_escalation', Reason):", supports('childA', 'needs_escalation', X))
print("supports_chain(childA,'needs_escalation', Y):", supports_chain('childA', 'needs_escalation', Y))



=== 1) EDB sanity check (raw facts) ===
age_years: X
-
6
temp_f: T    
-----
102.6
pain_severity: X               
----------------
mild_to_moderate

=== 2) IDB derivation (new facts from rules) ===
derived symptom(childA, ...): X             
--------------
no_tears      
dry_mucosa    
very_low_urine
low_urine     
no_cough      
abdominal_pain
derived symptom(childA,'abdominal_pain'): [()]

=== 3) OR-across-rules demonstration ===
abdominal_pain can come from mild_to_moderate OR severe
pain_severity(childA,'mild_to_moderate') -> abdominal_pain: [()]

=== 4) Stratified negation demonstration (no_cough) ===
no cough present in facts, so symptom(childA,'no_cough') derives:
no_cough: [()]

=== 5) Conjunction + numeric comparison (fever threshold) ===
needs_test(childA, X): X                     
----------------------
rapid_strep_or_culture

=== 6) A derived conclusion depending on another derived conclusion ===
antibiotic_plan(childA, X): X                                  
----------

In [ ]:

# ============================================================
# 12) UTILITIES: dose computation helper (kept separate from logic)
# ============================================================
def compute_dose_mg(med: str, weight_lb_val: float):
    """Return a dict of mg per dose and suggested interval/day caps (teaching-oriented)."""
    # Convert lb -> kg
    kg = weight_lb_val * 0.45359237
    # Pull KB parameters
    dose = dose_mg_per_kg(med, X).data[0][1]
    interval = interval_hours(med, X).data[0][1]
    max_day = max_mg_per_kg_day(med, X).data[0][1]
    max_doses = max_doses_per_day(med, X).data[0][1]
    mg_per_dose = kg * dose
    max_mg_day = kg * max_day
    return {
        "med": med,
        "weight_lb": weight_lb_val,
        "weight_kg": kg,
        "mg_per_dose": mg_per_dose,
        "interval_hours": interval,
        "max_mg_day": max_mg_day,
        "max_doses_per_day": max_doses,
    }

# ============================================================
# 13) DEMO QUERIES 
# ============================================================
def summarize_patient(p: str):
    print("\n============================================================")
    print(f"PATIENT: {p}")
    print("============================================================")

    print("\n--- Derived assessments ---")
    print("dehydration_level:", dehydration_level(p, Level))
    print("has_red_flag:", bool(has_red_flag(p)))
    print("escalation_level:", escalation_level(p, Level))
    print("likely_appendicitis_pattern:", bool(likely_appendicitis_pattern(p)))
    print("likely_viral_gi_pattern:", bool(likely_viral_gi_pattern(p)))

    print("\n--- Actions (with hard constraints) ---")
    print("recommended_action:", recommended_action(p, Action))
    print("forbidden_action:", forbidden_action(p, Action))

    print("\n--- Medication logic ---")
    print("risk_state:", risk_state(p, Cond))
    print("avoid_med:", avoid_med(p, Med))
    print("recommended_med:", recommended_med(p, Med))

    if weight_lb(p, X):
        w = weight_lb(p, X).data[0][1]
        med = 'acetaminophen' if recommended_med(p, 'acetaminophen') else 'ibuprofen'
        print("\nDose helper (mg) for:", med)
        print(compute_dose_mg(med, w))

    print("\n--- Testing / antibiotics (scoped) ---")
    print("needs_test:", needs_test(p, Test))
    print("antibiotic_plan:", antibiotic_plan(p, Val))

    print("\n--- Explanation snippets ---")
    print("explain_line(escalation):", explain_line(p, 'escalation', Reason))
    print("explain_line(dehydration):", explain_line(p, 'dehydration', Reason))
    print("explain_line(appendicitis):", explain_line(p, 'appendicitis', Reason))

# Run summaries
summarize_patient('childA')
summarize_patient('childB')
summarize_patient('childC')




In [26]:
# ============================================================
# 14) MULTI-TURN UPDATE DEMO (conversation evolves)
#    Add a new fact and re-query: childA now has worse urine timing
# ============================================================
print("\n\n================ Multi-turn update demo ================")
print("Before update:", hours_since_last_urine('childA', X))
# retract not shown; we add an additional fact to simulate new info:
+hours_since_last_urine('childA', 11)  # new info arrives
print("After update (facts present):", hours_since_last_urine('childA', X))
print("Now symptom very_low_urine:", bool(symptom('childA', 'very_low_urine')))
print("Now escalation_level:", escalation_level('childA', Level))
print("Forbidden home plan now?:", bool(forbidden_action('childA', 'home_plan_with_monitoring')))
print("Explanation escalation lines:", explain_line('childA', 'escalation', Reason))



================ Multi-turn update demo ================
Before update: X
-
7
After update (facts present): X 
--
7 
11
Now symptom very_low_urine: True
Now escalation_level: Level 
------
urgent
Forbidden home plan now?: True
Explanation escalation lines: Reason        
--------------
very_low_urine


In [38]:
# ORDERING 1 (good mental model): define the positive layer first, then the negated layer, then query.

pyDatalog.clear()

# EDB
+case('childA')
+symptom('childA', 'cough')

# Stratum 1: positive definition
_ = cough(P) <= symptom(P, 'cough')

# Stratum 2: stratified negation (only negates cough/1, which is already defined)
_ = no_cough(P) <= case(P) & ~cough(P)

# downstream decision
_ = eligible_home_care(P) <= case(P) & no_cough(P)

print("Ordering 1:")
print("  cough(childA)            =", cough('childA'))
print("  no_cough(childA)         =", no_cough('childA'))
print("  eligible_home_care(childA) =", eligible_home_care('childA'))


Ordering 1:
  cough(childA)            = [()]
  no_cough(childA)         = []
  eligible_home_care(childA) = []


In [39]:
# ORDERING 2 (shows the pitfall): write the negated layer first, query,
# THEN introduce the positive definition that the negation depended on.
# Same rules, different *rule introduction / query timing* → different observed conclusions.

pyDatalog.clear()

# EDB
+case('childA')
+symptom('childA', 'cough')   # cough is actually present

# Stratum 2 written first (depends on cough/1, which is not defined yet)
_ = no_cough(P) <= case(P) & ~cough(P)
_ = eligible_home_care(P) <= case(P) & no_cough(P)

print("Ordering 2 (before defining cough/1):")
print("  no_cough(childA)         =", no_cough('childA'))
print("  eligible_home_care(childA) =", eligible_home_care('childA'))

# Now introduce the (lower-stratum) positive definition
_ = cough(P) <= symptom(P, 'cough')

print("\nOrdering 2 (after defining cough/1):")
print("  cough(childA)            =", cough('childA'))
print("  no_cough(childA)         =", no_cough('childA'))
print("  eligible_home_care(childA) =", eligible_home_care('childA'))


Ordering 2 (before defining cough/1):
  no_cough(childA)         = 

AttributeError: Predicate without definition (or error in resolver): cough/1